# Diffie-Hellman and Cryptographic Randomness

These examples use the running scenario of St. Isidore Hospital. They are teaching examples: understand the mechanism, then prefer well-reviewed libraries and current protocols in production.

## Goal

Diffie-Hellman lets two parties derive the same shared secret over an insecure network. This notebook uses a small finite-field example for visibility; real systems use much larger approved parameters. Randomness supplies private exponents, nonces, and session keys. Weak randomness weakens everything built on top.

In [ ]:
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
import os, secrets, random

# Small teaching parameters: use large approved groups in production.
p = 23  # Public prime modulus.
g = 5   # Public generator.

# Each side chooses a private exponent and never sends it.
doctor_private = secrets.randbelow(p - 2) + 1
hospital_private = secrets.randbelow(p - 2) + 1

# Public values can travel over the network.
doctor_public = pow(g, doctor_private, p)
hospital_public = pow(g, hospital_private, p)

# Each side combines its private exponent with the other side's public value.
doctor_shared = pow(hospital_public, doctor_private, p)
hospital_shared = pow(doctor_public, hospital_private, p)
print(doctor_shared == hospital_shared)

session_key = HKDF(
    algorithm=hashes.SHA256(),
    length=32,
    salt=None,
    info=b"St Isidore VPN session",  # Context string binds the derived key to this purpose.
).derive(doctor_shared.to_bytes(2, "big"))  # Convert the shared number to bytes for the KDF.
print(session_key.hex())

In [ ]:
print("Cryptographic token:", secrets.token_hex(16))  # Use secrets for security tokens.

random.seed(42)  # A fixed seed makes random predictable.
print("Predictable token:", hex(random.getrandbits(128)))
random.seed(42)  # Resetting the seed repeats the same output.
print("Same predictable token:", hex(random.getrandbits(128)))

print("OS randomness:", os.urandom(16).hex())  # OS randomness is suitable for crypto seeds/keys.